In [ ]:
import pandas as pd
import numpy as np

'''
데이터 로드 + `제거
'''
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/'+ txt_dir, sep='|', dtype = str, engine='python')
    df.columns = [str(c).strip().strip("`") for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip("`")
    return df


In [ ]:
'''
월별 데이터 로드 함수 -> 다른 월에도 재사용 가능
'''

def load_month(month_str):
    return (
        load_csv(f'1. 주택기본정보/jtbasicinfo_{month_str}.txt'),
        load_csv(f'3. 전세임대 예측시세/depo_hosiseinfo_{month_str}.txt'),
        load_csv(f'5. 월세임대 예측시세/rent_hosiseinfo_{month_str}.txt'),
    )

df_주택기본정보, df_전세예측, df_월세예측 = load_month('202208')

In [ ]:
# 앤 필요없음
df_주택기본정보 = load_csv('1. 주택기본정보/jtbasicinfo_202208.txt')
df_전세예측 = load_csv('3. 전세임대 예측시세/depo_hosiseinfo_202208.txt')
df_월세예측 = load_csv('5. 월세임대 예측시세/rent_hosiseinfo_202208.txt')

In [ ]:
ref_year = 2022
LOW_Q = 0.2
old_threshold = 30

def prepare_price_ratio(df, price_col, dong_code_col, label):
    '''
    전세/월세 각각에 대해
    하위 분위 기준 계산
    저가여부 확인자 생성
    행정동별 저가 비율 및 건수 계산
    '''
    tmp = df.copy()
    tmp = tmp[tmp[price_col].notna()].copy()
    q = tmp[price_col].quantile(LOW_Q)
    # 저가 indicator
    tmp[f'is_low_{label}'] = (tmp[price_col] <= q).astype(int)

    # 동별 집계
    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_obs=(price_col, 'size'), low_ratio=(f'is_low_{label}', 'mean')).reset_index())

    agg = agg.rename(columns={'n_obs':f'n_{label}', 'low_ratio':f'low_ratio_{label}'})
    return agg, q


def prepare_old_ratio(df, build_year_col, dong_code_col):
    '''
    건축연도 기준 노후 비율 계산
    '''
    tmp = df.copy()
    tmp['SYDATE_parsed']= pd.to_datetime(tmp['SYDATE'], errors='coerce')
    tmp['build_year_final']=tmp['SYDATE_parsed'].dt.year
    
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp.loc[tmp['build_year_final']==0, 'build_year_final']=np.nan
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp['building_age'] = ref_year - tmp['build_year_final'].astype(int)
    tmp['is_old'] = (tmp['building_age'] >= old_threshold).astype(int)

    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_building=('building_age', 'size'), old_ratio=('is_old', 'mean')).reset_index())

    return agg

In [ ]:
'''
PNU코드의 앞 10자리와 뒤에 법정동 코드까지 이어지는 10자리 합이 같은지 체크 -> 동일
'''
df_주택기본정보['PNU'] = df_주택기본정보['PNU'].astype(str).str.strip()
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_주택기본정보['SREG'] = df_주택기본정보['SREG'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['SEUB'] = df_주택기본정보['SEUB'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['bjdong_code_check'] = df_주택기본정보['SREG']+df_주택기본정보['SEUB']
mismatch = df_주택기본정보[df_주택기본정보['bjdong_code'] != df_주택기본정보['bjdong_code_check']]
print(len(mismatch))

In [ ]:
'''
법정동 코드 추출
'''
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_전세예측['bjdong_code'] = df_전세예측['PNU'].str[:10]
df_월세예측['bjdong_code'] = df_월세예측['PNU'].str[:10]


## 행정동 코드 매핑

In [ ]:
'''
4/8 빅캠 방문 시 무조건 해야 할 코드
행정동 코드 매핑
법정동 코드(bjdong_code) → 행정동 코드(hdong_code), 행정동명(hdong_name)
형식 보고 전처리 필요할 수 있음 (예: 공백 제거, 문자열 정리 등)
'''

# 연계표 로드
df_mapping = pd.read_excel('행정동_법정동_연계표.xls', dtype=str)
# 컬럼명 확인 후 rename
# df_mapping.columns → ['bjdong_code', 'hdong_code', 'hdong_name', ...] 형태일 것

In [ ]:
'''
bjdong_code가 10자리인지, 앞뒤 공백 있는지 확인 후 전처리하고 merge
'''
print(df_mapping.columns.tolist())   # 이거 먼저 확인
print(df_mapping.head(3))            # 실제 값 형태 확인

In [ ]:
# row 단위 매핑
df_주택기본정보 = df_주택기본정보.merge(df_mapping[['bjdong_code','hdong_code','hdong_name']], on='bjdong_code', how='left')
df_전세예측     = df_전세예측.merge(df_mapping[['bjdong_code','hdong_code','hdong_name']], on='bjdong_code', how='left')
df_월세예측     = df_월세예측.merge(df_mapping[['bjdong_code','hdong_code','hdong_name']], on='bjdong_code', how='left')

# 매핑 누략률 확인
print("주택기본정보 매핑 누락률:", df_주택기본정보['hdong_code'].isna().mean())
print("전세예측 매핑 누락률:", df_전세예측['hdong_code'].isna().mean())
print("월세예측 매핑 누락률:", df_월세예측['hdong_code'].isna().mean())

### 이제부터 집계 단위는 모두 hdong_code

In [ ]:
dong_code_col = 'hdong_code' # 이후 분석에서는 행정동 코드 사용
price_col_depo = 'DEPO_PRED'
price_col_rent = 'PRED_RENT'
build_year_col = 'SYDATE'

In [ ]:
df_주택기본정보['BUILDYEAR']  = pd.to_numeric(df_주택기본정보['BUILDYEAR'], errors='coerce')
df_전세예측[price_col_depo] = pd.to_numeric(df_전세예측[price_col_depo], errors='coerce')
df_월세예측[price_col_rent] = pd.to_numeric(df_월세예측[price_col_rent], errors='coerce')

## 5-1. 저가 주거지 비율

In [ ]:
# 전세
depo_agg, depo_q = prepare_price_ratio(df_전세예측, 'DEPO_PRED', 'hdong_code', label='depo')
# 월세
rent_agg, rent_q = prepare_price_ratio(df_월세예측, 'PRED_RENT', 'hdong_code', label='rent')

## 5-2. 노후도 비율

In [ ]:
old_agg = prepare_old_ratio(df_주택기본정보, 'SYDATE', 'hdong_code')
# 기준: 2022 - 건축연도 >= 30년

## 5-3. 연립-다세대 집적도 (신규 추가)

In [ ]:
df_주택기본정보['SEDECNT'] = pd.to_numeric(df_주택기본정보['SEDECNT'], errors='coerce')

density_agg = (
    df_주택기본정보.groupby('hdong_code', dropna=False)
    .agg(
        n_building=('PKCODE1', 'nunique'),  # 건물 수
        total_units=('SEDECNT', 'sum'),      # 총 세대수
    )
    .reset_index()
)

# 6. HVI score

## 6-1. 3개 축 merge

In [ ]:
hvi = pd.merge(depo_agg, rent_agg, on='hdong_code', how='outer')
hvi = pd.merge(hvi, old_agg, on='hdong_code', how='left')
hvi = pd.merge(hvi, density_agg, on='hdong_code', how='left')

hvi['n_depo']=hvi['n_depo'].fillna(0)
hvi['n_rent']=hvi['n_rent'].fillna(0)
hvi['low_ratio_depo'] = hvi['low_ratio_depo'].fillna(np.nan)
hvi['low_ratio_rent'] = hvi['low_ratio_rent'].fillna(np.nan)

In [ ]:
def weighted_low_ratio(row):
    n_depo = row['n_depo']
    n_rent = row['n_rent']
    total = n_depo + n_rent
    if total == 0: return np.nan
    val = 0
    if pd.notna(row['low_ratio_depo']):
        val+= n_depo*row['low_ratio_depo']
    if pd.notna(row['low_ratio_rent']):
        val += n_rent*row['low_ratio_rent']
    return val/total

hvi['low_ratio_combined'] = hvi.apply(weighted_low_ratio, axis =1)

In [ ]:
hvi['score_low']     = pd.qcut(hvi['low_ratio_combined'], q=5, labels=[1,2,3,4,5], duplicates = 'drop').astype(float)
hvi['score_old']     = pd.qcut(hvi['old_ratio'],           q=5, labels=[1,2,3,4,5], duplicates = 'drop').astype(float)
hvi['score_density'] = pd.qcut(hvi['total_units'],         q=5, labels=[1,2,3,4,5], duplicates = 'drop').astype(float)

# 합산 (최대 15점)
hvi['HVI_score'] = hvi['score_low'] + hvi['score_old'] + hvi['score_density']
hvi['HVI_rank']  = hvi['HVI_score'].rank(ascending=False, method='min')
hvi['HVI_grade'] = pd.qcut(hvi['HVI_score'], q=5, labels=['매우낮음','낮음','보통','높음','매우높음'])

## 6-3. Sanity Check

- [ ]  HVI 상위 동 확인 → 도봉·노원·중랑·관악 등 나오면 정상
- [ ]  강남·서초가 상위면 기준값 재검토
- [ ]  결측 비율 확인 → `hvi.isna().mean()`

# 7. 전체 시간대 처리

In [ ]:
# 최근 3개월 중앙값 사용 (2022-06 ~ 2022-08)
# 이상치 영향 줄이기 위해 평균보다 중앙값이 안전
target_months = ['202206', '202207', '202208']

# STEP 8 — 반출용 결과 저장

> **반출 규정**: 빅밸류 데이터는 응용집계·시각화만 반출 가능. row 단위 원본 절대 포함 금지.

In [ ]:
export_hvi = hvi[[
    'hdong_code', 'hdong_name',
    'low_ratio_combined',  # 저가 주거지 비율 (집계값)
    'old_ratio',           # 노후도 비율 (집계값)
    'total_units',         # 총 세대수 (집계값)
    'HVI_score',
    'HVI_rank',
    'HVI_grade'
]].copy()

export_hvi.to_csv('export_hvi_hdong.csv', index=False, encoding='utf-8-sig')